In [1]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
import joblib
import json
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# 1. Load data files
events_df = pd.read_csv('../data/events.csv')
category_tree_df = pd.read_csv('../data/category_tree.csv')

# Process Category Tree
category_tree_df['parentid'] = category_tree_df['parentid'].fillna(0).astype(int)
category_map = dict(zip(category_tree_df['categoryid'], category_tree_df['parentid']))

# 2. Filter out inactive users and unpopular items
user_counts = events_df['visitorid'].value_counts()
item_counts = events_df['itemid'].value_counts()

active_users = user_counts[user_counts >= 5].index
active_items = item_counts[item_counts >= 10].index

df_filtered = events_df[
    (events_df['visitorid'].isin(active_users)) & 
    (events_df['itemid'].isin(active_items))
].copy()

# 3. Map implicit event weights
event_weights = {
    'view': 1.0,
    'addtocart': 3.0,
    'transaction': 5.0
}
df_filtered['interaction_score'] = df_filtered['event'].map(event_weights)

# 4. Aggregate multiple interactions per (visitor, item) pair
user_item_df = df_filtered.groupby(['visitorid', 'itemid'])['interaction_score'].sum().reset_index()

print(f"Cleaned dataset: {len(user_item_df)} unique interactions across {user_item_df['visitorid'].nunique()} users and {user_item_df['itemid'].nunique()} items.")
print(f"Loaded {len(category_tree_df)} categories from category_tree.csv.")

Cleaned dataset: 458786 unique interactions across 80123 users and 47431 items.
Loaded 1669 categories from category_tree.csv.


In [3]:
# Create sparse pivot matrix (Users as rows, Items as columns)
user_ids = user_item_df['visitorid'].astype('category')
item_ids = user_item_df['itemid'].astype('category')

user_item_matrix = sp.csr_matrix(
    (user_item_df['interaction_score'], (user_ids.cat.codes, item_ids.cat.codes))
)

# Compute Item-Item Cosine Similarity
item_similarity_matrix = cosine_similarity(user_item_matrix.T, dense_output=False)

# Build a lookup dictionary mapping unique item_id -> top 10 recommended items
unique_items = item_ids.cat.categories
similarity_lookup = {}

for idx, item_id in enumerate(unique_items):
    sim_scores = item_similarity_matrix[idx].toarray().flatten()
    top_indices = np.argsort(sim_scores)[::-1][1:11]
    
    similarity_lookup[int(item_id)] = [
        {
            "recommended_item_id": int(unique_items[i]),
            "similarity_score": float(round(sim_scores[i], 4))
        }
        for i in top_indices if sim_scores[i] > 0
    ]

print("Cosine Similarity Lookup Table built successfully!")

Cosine Similarity Lookup Table built successfully!


In [4]:
# Feature Engineering for Random Forest & Logistic Regression
events_features = df_filtered.copy()
events_features['is_conversion'] = (events_features['event'] == 'transaction').astype(int)

# Aggregate user-level and item-level stats
user_stats = events_features.groupby('visitorid').agg(
    user_total_events=('event', 'count'),
    user_total_score=('interaction_score', 'sum')
).reset_index()

item_stats = events_features.groupby('itemid').agg(
    item_total_views=('event', lambda x: (x == 'view').sum()),
    item_total_carts=('event', lambda x: (x == 'addtocart').sum()),
    item_total_purchases=('event', lambda x: (x == 'transaction').sum())
).reset_index()

# Merge features
feature_df = events_features[['visitorid', 'itemid', 'interaction_score', 'is_conversion']].drop_duplicates()
feature_df = feature_df.merge(user_stats, on='visitorid', how='left')
feature_df = feature_df.merge(item_stats, on='itemid', how='left')

X = feature_df[['interaction_score', 'user_total_events', 'user_total_score', 'item_total_views', 'item_total_carts', 'item_total_purchases']]
y = feature_df['is_conversion']

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train Logistic Regression Baseline
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

# Train Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate Models
rf_preds = rf_model.predict_proba(X_test)[:, 1]
print("Random Forest ROC-AUC Score:", roc_auc_score(y_test, rf_preds))

Random Forest ROC-AUC Score: 1.0


In [ ]:
# Ensure directories exist
os.makedirs('../saved_models', exist_ok=True)
os.makedirs('../data', exist_ok=True)

# 1. Save trained ML models (.pkl)
joblib.dump(rf_model, '../saved_models/random_forest_conversion.pkl')
joblib.dump(lr_model, '../saved_models/logistic_baseline.pkl')

# 2. Save Similarity Lookup JSON
with open('../data/similarity_lookup.json', 'w') as f:
    json.dump(similarity_lookup, f)

# 3. Create & Save Product Catalog JSON with category mapping
category_list = list(category_map.keys())
sample_products = []

for item_id in list(unique_items[:150]):
    cat_id = int(np.random.choice(category_list))
    parent_cat_id = int(category_map.get(cat_id, 0))
    
    sample_products.append({
        "item_id": int(item_id),
        "name": f"Retail Item #{item_id}",
        "price": float(round(np.random.uniform(19.99, 299.99), 2)),
        "category_id": cat_id,
        "parent_category_id": parent_cat_id,
        "rating": float(round(np.random.uniform(3.8, 5.0), 1))
    })

with open('../data/processed_products.json', 'w') as f:
    json.dump(sample_products, f)

# 4. Save Sample User Profiles JSON (for UI profile switcher)
sample_user_ids = [int(uid) for uid in user_item_df['visitorid'].unique()[:10]]
with open('../data/sample_users.json', 'w') as f:
    json.dump(sample_user_ids, f)

print("Phase 1 artifacts successfully exported!")